# OEC LTX-2.3 — model-server Gradio notebook

This notebook is a thin client; it does not import LTX, PyTorch, xFormers, or model weights into the Jupyter kernel. Text-to-video and image-to-video run through the isolated \`ltx-api\` service in the \`phase1_ltx\` environment.

Start the service from the OEC repository root on an admitted idle A100:

\`\`\`bash
python tools/oec_service_manager.py start ltx-api --gpu <ID> --port 7869 --allow-download
python tools/oec_service_manager.py status ltx-api
\`\`\`

Cache fills are optional but must pass a fresh StorageManager admission. Do not disable quota guards. The service binds to loopback; use an SSH tunnel for remote Jupyter. Gradio sharing is disabled.

The notebook exposes both modes:
- Text-to-video: prompt → MP4
- Image-to-video: reference image + prompt → MP4

Use the low-resolution presets for routine smoke tests. A successful UI launch alone is not a model test; the callback waits for a real API job and verifies the MP4 stream.

In [ ]:
import core_config

core_config.map_storage()
core_config.set_hpc_proxies()
try:
    core_config.authenticate_huggingface(interactive=False)
except Exception as exc:
    print(f"[INFO] Non-interactive Hugging Face auth unavailable: {type(exc).__name__}")
print("[SYSTEM] OEC storage mapping and non-interactive authentication check complete.")

In [ ]:
# Read-only process inspection. Never kill a hard-coded PID on a shared host.
import os
import subprocess

user = os.environ.get("USER", "")
if user:
    result = subprocess.run(
        ["ps", "-u", user, "-o", "pid,lstart,etime,comm"],
        capture_output=True, text=True, check=False,
    )
    print(result.stdout)
    if result.returncode:
        print(result.stderr[:1000])
else:
    print("USER is unavailable; skipped process-list inspection.")

In [ ]:
!nvidia-smi

In [ ]:
# Read-only frontend dependency checks. No package installs/upgrades in this notebook.
import importlib.metadata
import shutil
import sys

for package in ("gradio", "gradio_client", "requests", "Pillow"):
    try:
        print(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(
            f"{package!r} is missing from {sys.executable}; select the configured phase1_ltx kernel."
        ) from exc
if shutil.which("ffprobe") is None:
    raise RuntimeError("ffprobe is required for decoded MP4 validation.")
import gradio as gr
import requests
from PIL import Image
print("LTX_NOTEBOOK_CLIENT_DEPENDENCIES=PASS")

In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path
import shutil
import struct
import subprocess
import time
import urllib.parse

import gradio as gr
import requests
from PIL import Image

LTX_API_BASE = os.environ.get("OEC_LTX_SERVICE_URL", os.environ.get("OEC_MEDIA_API_URL", "http://127.0.0.1:7869")).rstrip("/")
LTX_RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
LTX_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if urllib.parse.urlsplit(LTX_API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("LTX notebook only permits a loopback model-server endpoint.")
LTX_HTTP = requests.Session()
LTX_HTTP.trust_env = False


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _ffprobe_mp4(path: Path) -> dict:
    if not path.is_file() or path.is_symlink() or path.stat().st_size <= 1024:
        raise gr.Error(f"LTX returned a missing or too-small MP4: {path}")
    if len(path.read_bytes()[:16]) < 12 or path.read_bytes()[4:8] != b"ftyp":
        raise gr.Error("LTX result does not contain an ISO-BMFF/MP4 file header.")
    probe = shutil.which("ffprobe")
    if not probe:
        raise gr.Error("ffprobe is required to validate a generated LTX MP4.")
    completed = subprocess.run(
        [probe, "-v", "error", "-count_frames", "-select_streams", "v:0",
         "-show_entries", "stream=codec_name,width,height,nb_read_frames,r_frame_rate,duration",
         "-show_entries", "format=duration,format_name", "-of", "json", str(path)],
        text=True, capture_output=True, timeout=60, check=False,
    )
    if completed.returncode:
        raise gr.Error(f"ffprobe rejected the LTX MP4: {completed.stderr[-1000:]}")
    report = json.loads(completed.stdout)
    streams = report.get("streams") or []
    if not streams:
        raise gr.Error("LTX MP4 has no video stream.")
    stream = streams[0]
    try:
        width, height = int(stream.get("width") or 0), int(stream.get("height") or 0)
        frames = int(stream.get("nb_read_frames") or 0)
        duration = float((report.get("format") or {}).get("duration") or stream.get("duration") or 0.0)
    except (TypeError, ValueError) as exc:
        raise gr.Error("LTX MP4 stream metadata is malformed.") from exc
    if width <= 0 or height <= 0 or frames < 5 or duration <= 0:
        raise gr.Error(f"LTX MP4 stream metadata is invalid: {stream}")
    return {
        "bytes": path.stat().st_size,
        "sha256": _sha256_file(path),
        "format": (report.get("format") or {}).get("format_name"),
        "codec": stream.get("codec_name"),
        "width": width, "height": height, "frames": frames,
        "fps": stream.get("r_frame_rate"), "duration_seconds": duration,
    }


def ltx_service_health():
    try:
        response = LTX_HTTP.get(f"{LTX_API_BASE}/health", timeout=8)
        health = response.json()
    except Exception as exc:
        raise gr.Error(
            f"LTX model server is unreachable at {LTX_API_BASE}. Start ltx-api and inspect its service logs."
        ) from exc
    if response.status_code != 200 or not health.get("ready") or not health.get("model_loaded"):
        raise gr.Error(f"LTX model server is not ready: {json.dumps(health, ensure_ascii=False)}")
    return health


def generate_video(task, prompt, reference_image, duration, fps, seed, randomize_seed, enhance_prompt, resolution):
    if task not in {"text_to_video", "image_to_video"}:
        raise gr.Error("Choose text-to-video or image-to-video.")
    if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > 800:
        raise gr.Error("Enter a non-empty prompt of at most 800 characters.")
    try:
        width, height = (int(value) for value in str(resolution).lower().split("x", 1))
    except Exception as exc:
        raise gr.Error("Choose a supported resolution.") from exc
    if (width, height) not in {(768, 512), (512, 768), (768, 768), (1024, 1024), (1024, 1536), (1536, 1024)}:
        raise gr.Error("Unsupported LTX resolution pair.")
    if not 1 <= float(duration) <= 10 or not 12 <= float(fps) <= 30:
        raise gr.Error("Duration must be 1–10 seconds and FPS 12–30.")
    seed = int(seed)
    if not 0 <= seed <= 2_147_483_647:
        raise gr.Error("Seed must be between 0 and 2147483647.")

    health = ltx_service_health()
    payload = {
        "task": task,
        "prompt": prompt.strip(),
        "parameters": {
            "duration_seconds": float(duration), "fps": int(fps), "seed": seed,
            "randomize_seed": bool(randomize_seed), "enhance_prompt": bool(enhance_prompt),
            "width": width, "height": height,
        },
    }
    if task == "image_to_video":
        if not reference_image:
            raise gr.Error("Upload a reference image for image-to-video mode.")
        image_path = Path(str(reference_image)).expanduser().resolve()
        if not image_path.is_file() or image_path.is_symlink():
            raise gr.Error("Reference image path is missing or unsafe.")
        try:
            with Image.open(image_path) as reference:
                if reference.format not in {"PNG", "JPEG", "WEBP"}:
                    raise ValueError(f"unsupported image format: {reference.format}")
                reference.verify()
        except Exception as exc:
            raise gr.Error("Reference must be a valid PNG, JPEG, or WEBP image.") from exc
        raw_image = image_path.read_bytes()
        if not raw_image or len(raw_image) > 12 * 1024 * 1024:
            raise gr.Error("Reference image must be non-empty and no larger than 12 MiB.")
        payload["filename"] = image_path.name
        payload["image_base64"] = base64.b64encode(raw_image).decode("ascii")

    try:
        accepted_response = LTX_HTTP.post(f"{LTX_API_BASE}/v1/jobs", json=payload, timeout=30)
    except Exception as exc:
        raise gr.Error(f"LTX job submission failed: {type(exc).__name__}") from exc
    if accepted_response.status_code != 202:
        raise gr.Error(f"LTX job rejected (HTTP {accepted_response.status_code}): {accepted_response.text[:1000]}")
    accepted = accepted_response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected LTX job response: {accepted}")
    deadline = time.monotonic() + 1800
    latest = accepted
    while time.monotonic() < deadline:
        poll = LTX_HTTP.get(f"{LTX_API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"LTX job status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"LTX job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("LTX generation timed out; inspect ltx-api logs.")

    result_response = LTX_HTTP.get(f"{LTX_API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result_response.status_code != 200:
        raise gr.Error(f"LTX artifact download failed (HTTP {result_response.status_code}): {result_response.text[:800]}")
    content_type = result_response.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type != "video/mp4":
        raise gr.Error(f"Expected video/mp4 from LTX, received {content_type!r}.")
    output_path = LTX_RESULT_ROOT / f"{job_id}.mp4"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing LTX notebook result.")
    output_path.write_bytes(result_response.content)
    try:
        validation = _ffprobe_mp4(output_path)
    except Exception:
        output_path.unlink(missing_ok=True)
        raise
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": LTX_API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(output_path), "video_validation": validation,
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)


with gr.Blocks(title="LTX-2.3 — text/image to video", theme=gr.themes.Monochrome()) as demo_ltx:
    gr.Markdown("# LTX-2.3 — text-to-video and image-to-video")
    gr.Markdown("The isolated ltx-api service owns model loading. This UI submits jobs, retrieves the MP4, and validates its decoded video stream.")
    with gr.Row():
        with gr.Column():
            task = gr.Radio(["text_to_video", "image_to_video"], value="text_to_video", label="Generation mode")
            prompt = gr.Textbox(label="Prompt", value="A miniature brass robot walking across a studio table; gentle dolly camera", lines=3)
            reference_image = gr.Image(type="filepath", label="Reference image (image-to-video only)")
            duration = gr.Slider(1, 10, value=1, step=1, label="Duration (seconds)")
            fps = gr.Slider(12, 30, value=12, step=3, label="Frames per second")
            seed = gr.Number(value=42, precision=0, label="Seed")
            randomize_seed = gr.Checkbox(value=False, label="Randomize seed")
            enhance_prompt = gr.Checkbox(value=False, label="Enhance prompt")
            resolution = gr.Dropdown(["768x512", "512x768", "768x768", "1024x1024", "1024x1536", "1536x1024"],
                                     value="768x512", label="Resolution")
            generate_button = gr.Button("Generate video", variant="primary")
        with gr.Column():
            output_video = gr.Video(label="Generated MP4")
            output_metadata = gr.Textbox(label="Server and MP4 metadata", lines=14)
    generate_button.click(
        fn=generate_video,
        inputs=[task, prompt, reference_image, duration, fps, seed, randomize_seed, enhance_prompt, resolution],
        outputs=[output_video, output_metadata],
    )

if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo_ltx.queue(max_size=1).launch(
        server_name="127.0.0.1", server_port=7869, share=False, inline=False, show_error=True, allowed_paths=[str(LTX_RESULT_ROOT)]
    )

In [ ]:
# The acceptance harness closes its own UI on exit; do not keep demo handles alive here.

In [ ]:
!nvidia-smi